In [16]:
import pandas as pd
pd.set_option("display.max_rows", None)

In [3]:
log = pd.read_csv("scrape_log.csv")

In [4]:
log

,fixture_id,status,http_status,file_name,error,attempts,timestamp
0,70600,success,200,fixture_70600.json,NaN,1.0,2026-07-21T16:22:23-04:00
1,71800,failed_500,500,NaN,<!DOCTYPE html>\n<html>\n <head>\n <...,3.0,2026-07-21T16:22:23-04:00
2,70598,success,200,fixture_70598.json,NaN,1.0,2026-07-21T16:23:55-04:00
3,70599,success,200,fixture_70599.json,NaN,1.0,2026-07-21T16:23:57-04:00
4,70601,success,200,fixture_70601.json,NaN,1.0,2026-07-21T16:24:00-04:00
...,...,...,...,...,...,...,...
23496,72496,failed_500,500,NaN,<!DOCTYPE html>\n<html>\n <head>\n <...,NaN,2026-07-22T12:37:31-04:00
23497,72497,failed_500,500,NaN,<!DOCTYPE html>\n<html>\n <head>\n <...,NaN,2026-07-22T12:37:39-04:00
23498,72498,failed_500,500,NaN,<!DOCTYPE html>\n<html>\n <head>\n <...,NaN,2026-07-22T12:37:46-04:00
23499,72499,failed_500,500,NaN,<!DOCTYPE html>\n<html>\n <head>\n <...,NaN,2026-07-22T12:37:54-04:00


### Data Collection Report

In [6]:
#checking status
log['status'].value_counts()
# 2474 are failed_500. None are incomplete or unexpected in structure.

status
success       21027
failed_500     2474
Name: count, dtype: int64



| Metric                |  Value |
| --------------------- | -----: |
| Fixture IDs attempted | 23,501 |
| Successful JSONs      | 21,027 |
| Failed (500)          |  2,474 |
| Success rate          |  89.5% |


### Inspecting HTTP 500 entries

In [8]:
log_failed = log[log['status'] == 'failed_500']

In [9]:
len(log_failed)

2474

In [10]:
log_failed['fixture_id']

1        71800
66       49055
67       49056
117      49106
118      49107
         ...  
23496    72496
23497    72497
23498    72498
23499    72499
23500    72500
Name: fixture_id, Length: 2474, dtype: int64

In [17]:
# finding consecutive chunks of failed responses and putting them into a data frame
failed_ids = sorted(log_failed["fixture_id"].tolist()) # sort failed ids in a list from small to large numbers
chunks = []

start = failed_ids[0]
prev = failed_ids[0]

for current in failed_ids[1:]:
    if current == prev + 1:
        prev = current
    else:
        chunks.append((start, prev, prev - start + 1))
        start = current
        prev = current

chunks.append((start, prev, prev - start + 1)) #to save the last iteration

chunks_df = pd.DataFrame(
    chunks,
    columns=["start", "end", "length"]
)

chunks_df.sort_values("length", ascending=False) #sort by longest chunk

#"Is this number exactly one more than the previous one?"
# Yes → we're still inside the same chunk, so extend it.
# No → the previous chunk is finished, save it and start a new one

,start,end,length
186,56044,56319,276
187,56326,56591,266
614,72297,72500,204
208,57762,57833,72
185,55976,56035,60
360,62177,62235,59
405,63971,64027,57
601,71963,72003,41
410,64083,64121,39
496,67457,67494,38


In [22]:
# checking cumulative share of each chunk to know how many I need to check to cover > 50% of this dataset
chunks_df = chunks_df.sort_values("length", ascending=False)

chunks_df["cum_failed"] = chunks_df["length"].cumsum()
chunks_df["cum_pct"] = (
    chunks_df["cum_failed"] / chunks_df["length"].sum() * 100
)

chunks_df.head(60)

,start,end,length,cum_failed,cum_pct
186,56044,56319,276,276,11.156023
187,56326,56591,266,542,21.907842
614,72297,72500,204,746,30.153597
208,57762,57833,72,818,33.063864
185,55976,56035,60,878,35.489086
360,62177,62235,59,937,37.873888
405,63971,64027,57,994,40.177850
601,71963,72003,41,1035,41.835085
410,64083,64121,39,1074,43.411479
496,67457,67494,38,1112,44.947454


### Getting only JSONs with HTTP status = success / 200

In [23]:
from pathlib import Path
import json

In [24]:
#keeping only those logs in scrape_log.csv where status is success
log_success = log[log["status"] == "success"]

In [25]:
# sanity check
len(log_success)

21027

In [29]:
# getting file names from the successful logs
success_files = log_success["file_name"].tolist()
success_files[1:10]

['fixture_70598.json',
 'fixture_70599.json',
 'fixture_70601.json',
 'fixture_70602.json',
 'fixture_70603.json',
 'fixture_70604.json',
 'fixture_70605.json',
 'fixture_70606.json',
 'fixture_70607.json']

In [32]:
# getting only successful jsons i.e. success_files in a list

json_folder = Path("./raw_match_centres") #from the root of this, go up till raw_match_centres

records = []

for file in success_files: #pick up the file names that have http status = success
    with open(json_folder / file, "r", encoding="utf-8") as f: #define their path!
        # / means join folder and filename. json_folder is data_collection/raw_match_centres and file is xxxx.json. so full path becomes data_collection/raw_match_centres/xxxxx.json.
        # open the file.
        # r means read only, don't edit it.
        # utf 8 is just some standard thing, ignore it.
        # f represents the existing open file. so with the opened file as f, append it to records.
        # json.load() converts file into python dictionary.
        records.append(json.load(f)) #add them to records!

In [33]:
# sanity check 
len(records)

21027

In [34]:
records[0].keys()

dict_keys(['fixture', 'teams_data', 'tournament', 'team1_players', 'team2_players', 'squad1_list', 'squad2_list', 'events', 'team1_score', 'team2_score', 'report', 'lineup1', 'lineup2', 'official1', 'official2', 'mvp', 'stats_available'])